# A chess value network: from master games to an opponent

This notebook is the full machine-learning workflow for PyChess's neural-network
opponent. It starts from raw game records and ends with an exported model that
plays chess.

1. **Data** — 10,000 recent over-the-board master games (both players rated 2300+),
   curated by `ai/data/download_games.py` from *The Week in Chess*.
2. **Exploration and cleaning** — what is in the games, and which ones to drop.
3. **Features** — every position of every game, encoded by the C++ engine as one-hot
   planes (pieces, castling rights, en passant, 50-move clock).
4. **Model** — a residual convolutional network (CNN) estimating, for any position,
   the probability that the side to move **wins, draws or loses**. Those map
   directly to *P(White wins)*, *P(draw)*, *P(Black wins)*.
5. **Training and evaluation** — a split by game, simple baselines, early stopping,
   calibration checks and an error analysis.
6. **Inference** — the model plays by picking the legal move that leads to the
   position with the best expected outcome for its colour. It is then tested in
   batched games against simple opponents.
7. **Export** — ONNX and `torch.export` files, plus a model card, for deployment.

**Where the work runs.** The chess logic runs in the repository's C++ engine
(`core/`), called from Python through pybind11 (`chess_engine`). That covers
replaying PGN, legal moves, encoding and game-over detection. On a GPU, the
engine's CUDA kernels encode positions and generate moves directly on PyTorch
tensors. The whole training set lives on the GPU as compact 72-byte positions,
and each mini-batch is encoded on the fly, so no CPU data loader sits in the loop.

**Running it.** In Google Colab, choose *Runtime → Change runtime type → GPU*,
then *Run all*. The first cell clones the repository and builds the engine, CUDA
kernels included, for the attached GPU. Locally, run `pip install .` in the
repository root first. A `PROFILE` setting sizes the run: `gpu` (default when a GPU
is present) trains the full model, and `cpu` trains a smaller one on sub-sampled
epochs.

> The outputs saved in this notebook come from the **`cpu` profile on a 4-core
> machine without a GPU**. Every step ran for real, but on a reduced model and
> budget, so treat the metrics as a lower bound. A Colab GPU run with the `gpu`
> profile produces the model meant for deployment.

## 0. Setup

In [ ]:
# Works on Google Colab (GPU runtime) and locally from a clone of the repository.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/ziggityzaggity/Chess.git"
BRANCH = "claude/great-curie-tlvovw"   # the branch with the ML engine work; use "main" once merged
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/chess")
    if not ROOT.exists():
        url = REPO_URL
        try:  # private repository: store a GitHub token as the Colab secret GITHUB_TOKEN
            from google.colab import userdata
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(ROOT)], check=True)
    if pathlib.Path("/usr/local/cuda/bin/nvcc").exists():
        os.environ.setdefault("CUDACXX", "/usr/local/cuda/bin/nvcc")
    # Compiles the C++ engine and its CUDA kernels for this GPU (a minute or two);
    # onnxruntime is only used to verify the exported model at the end.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(ROOT), "onnxruntime"], check=True)
else:
    ROOT = pathlib.Path.cwd().resolve()
    while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    try:
        import chess_engine  # noqa: F401 — installed with `pip install .`
    except ImportError:      # or a CMake development build (see bindings/python/README.md)
        sys.path.insert(0, str(ROOT / "build" / "python"))
print("repository:", ROOT)

In [ ]:
import copy, inspect, json, math, random, time
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import chess_engine as ce
from chess_engine import gpu, player

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} on {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else f" ({os.cpu_count()} CPU threads)"))
print(f"chess_engine {ce.__version__}: {ce.cuda_build_info()}; GPU kernels usable: {gpu.kernels_available()}")
pd.set_option("display.precision", 4)

Every chart below uses one quiet style. Outcomes always use the same three colours:
**blue** is a win for White (or for the side to move), **grey** is a draw, and
**red** is a win for Black (or a loss for the side to move).

In [ ]:
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
WIN, DRAW, LOSS = "#2a78d6", "#898781", "#e34948"      # outcome polarity: blue / neutral / red
OUTCOME_COLORS = [WIN, DRAW, LOSS]
BLUES = mpl.colors.LinearSegmentedColormap.from_list("blues", [SURFACE, "#cde2fb", "#6da7ec", "#2a78d6", "#104281"])

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "font.size": 10,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 11.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "lines.linewidth": 2, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
    "axes.prop_cycle": mpl.cycler(color=SERIES), "legend.frameon": False, "legend.labelcolor": INK2,
})


def bar_labels(ax, bars, fmt="{:.0f}", inside=False):
    # Value at the tip of each bar, in ink (never in the bar's colour).
    for b in bars:
        v = b.get_height()
        ax.annotate(fmt.format(v), (b.get_x() + b.get_width() / 2, v), xytext=(0, 3),
                    textcoords="offset points", ha="center", va="bottom", color=INK2, fontsize=9)

### Configuration

All the knobs in one place. Positions within a game are strongly correlated, so an
"epoch" may use only a random sample of the training positions
(`samples_per_epoch`). On a CPU this keeps a run to minutes. On a GPU the full set
is cheap.

In [ ]:
@dataclass
class Config:
    seed: int = 7
    # data cleaning
    min_plies: int = 20            # drop games shorter than this (forfeits, opening accidents)
    min_draw_plies: int = 40       # drop draws agreed before move 20 ("grandmaster draws")
    max_games: int | None = None   # use only the first N games (for quick runs)
    val_frac: float = 0.10
    test_frac: float = 0.10
    # model
    channels: int = 64
    blocks: int = 6
    head_channels: int = 32
    hidden: int = 128
    dropout: float = 0.3
    # training
    epochs: int = 20
    samples_per_epoch: int | None = None   # None = every training position, every epoch
    batch_size: int = 1024
    lr: float = 2e-3
    weight_decay: float = 1e-4
    patience: int = 4              # early stopping: epochs without validation improvement
    mirror_augment: bool = True
    # evaluation games
    match_games: int = 200
    match_max_plies: int = 300
    opening_random_plies: int = 4


PROFILES = {
    "gpu": Config(),
    "cpu": Config(channels=32, blocks=3, epochs=8, samples_per_epoch=150_000, batch_size=512,
                  patience=3, match_games=64),
    "smoke": Config(max_games=400, channels=16, blocks=1, epochs=2, samples_per_epoch=5_000,
                    batch_size=256, match_games=8, match_max_plies=60),
}
PROFILE = os.environ.get("CHESS_PROFILE") or ("gpu" if DEVICE.type == "cuda" else "cpu")
cfg = PROFILES[PROFILE]


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything(cfg.seed)
print(f"profile: {PROFILE}\n" + ", ".join(f"{k}={v}" for k, v in asdict(cfg).items()))

### Engine self-test

The CUDA kernels and the CPU engine share one implementation of the chess rules,
but a GPU is a different machine, so we verify before trusting it. The test counts
the move trees of standard *perft* positions: every legal move sequence to a fixed
depth, against published totals. That exercises castling, en passant, promotions
and check. It then compares encoding and move generation on this device with the
CPU engine, element for element.

In [ ]:
assert gpu.self_test(DEVICE), "engine self-test failed on " + str(DEVICE)

## 1. The raw data

`ai/data/twic_otb2300.pgn` holds 10,000 games from *The Week in Chess* issues
1635–1649 (spring 2026). They were selected by `ai/data/download_games.py`, which
records its choices in `twic_otb2300.manifest.json`. The selection is "relevant"
in a specific sense: each game's result should reflect what happened on the board.

* **Over-the-board, classical time control only.** Online blitz (Titled
  Tuesday), rapid, blitz, Armageddon, engine and exhibition events are removed.
  Time scrambles decide many of those games regardless of the position.
* **Both players FIDE-rated 2300 or higher.** At that level a winning position
  is usually converted, so outcomes carry more signal about the position.
* **Standard chess from the initial position**, with a proper result and no
  duplicates.

In [ ]:
PGN = ROOT / "ai" / "data" / "twic_otb2300.pgn"
manifest = json.loads(PGN.with_suffix(".manifest.json").read_text())
print({k: manifest[k] for k in ("games", "source", "date_range")})
print("filters:", {k: v for k, v in manifest["filters"].items() if k != "excluded_event_pattern"})

raw = ce.split_pgn(PGN.read_text())          # [(tags, movetext)], parsed by the C++ engine
if cfg.max_games:
    raw = raw[: cfg.max_games]
games = pd.DataFrame([{**tags, "movetext": movetext} for tags, movetext in raw])
print(f"{len(games):,} games, {games.shape[1]} columns")
games.head(3)

### Data dictionary and types

PGN tags are strings. Convert the ones with meaning, like ratings and dates, and
add a few derived columns. Where values are missing, how many, and why matters
more than the cleaning itself.

In [ ]:
for col in ("WhiteElo", "BlackElo"):
    games[col] = pd.to_numeric(games[col], errors="coerce")
games["Date"] = pd.to_datetime(games["Date"].str.replace("??", "01"), format="%Y.%m.%d", errors="coerce")
games["white_score"] = games["Result"].map({"1-0": 1.0, "1/2-1/2": 0.5, "0-1": 0.0})
games["mean_elo"] = (games["WhiteElo"] + games["BlackElo"]) / 2
games["elo_diff"] = games["WhiteElo"] - games["BlackElo"]

missing = games.isna().mean().mul(100).round(1)
print("missing values (% of games), columns with any:")
print(missing[missing > 0].to_string())
games[["WhiteElo", "BlackElo", "mean_elo", "elo_diff", "white_score"]].describe().round(1)

Missing values sit in optional tags only: titles (not every player holds one),
`Variation`, and team fields that exist only for team events. The model never
uses them. Ratings and results, the columns used for selection and labels, are
complete. That is guaranteed by the download filter.

### What the games look like

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1, 1.4, 1.4]})

ax = axes[0]
counts = games["Result"].value_counts().reindex(["1-0", "1/2-1/2", "0-1"])
bars = ax.bar(["White wins", "Draw", "Black wins"], counts.values / len(games) * 100,
              color=OUTCOME_COLORS, width=0.6)
bar_labels(ax, bars, "{:.1f}%")
ax.set_title("Game results")
ax.set_ylabel("% of games")
ax.set_ylim(0, counts.max() / len(games) * 118)
ax.grid(axis="x", visible=False)

ax = axes[1]
ratings = pd.concat([games["WhiteElo"], games["BlackElo"]])
ax.hist(ratings, bins=np.arange(2300, ratings.max() + 25, 25), color=SERIES[0], rwidth=0.85)
ax.set_title("Player ratings (both colours)")
ax.set_xlabel("FIDE Elo")
ax.set_ylabel("games")

ax = axes[2]
bins = np.arange(-400, 401, 50)
cut = pd.cut(games["elo_diff"], bins)
grp = games.groupby(cut, observed=True)["white_score"].agg(["mean", "count", "std"])
mid = np.array([iv.mid for iv in grp.index])
err = 1.96 * grp["std"] / np.sqrt(grp["count"])
ax.errorbar(mid, grp["mean"], yerr=err, fmt="o-", color=SERIES[0], ms=4.5, capsize=0, lw=2,
            label="observed (95% CI)")
d = np.linspace(-400, 400, 200)
ax.plot(d, 1 / (1 + 10 ** (-d / 400)), color=MUTED, lw=1.5, label="Elo model")
ax.set_title("White's score vs rating advantage")
ax.set_xlabel("White Elo − Black Elo")
ax.set_ylabel("White's mean score")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

* **Results.** White wins about 30% of these games, Black about 21%, and about
  half are drawn. The model predicts all three outcomes, so draws stay in, and
  the imbalance is modest enough to need no re-weighting. Re-weighting would
  distort the probabilities we want to use.
* **Ratings.** Everyone is 2300+, with a long tail of 2600–2800 elite players.
* **Rating advantage predicts results**, and it roughly follows the classic Elo
  curve. The model never sees ratings, only the board. Part of what decides a
  game is therefore invisible to it: the irreducible noise in our labels.

In [ ]:
games["plies"] = [len(ce.san_tokens(m)[0]) for m in games["movetext"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1.5, 1]})
ax = axes[0]
edges = np.arange(0, 241, 6)
for res, label, color in [("1-0", "White wins", WIN), ("1/2-1/2", "Draw", DRAW), ("0-1", "Black wins", LOSS)]:
    ax.hist(games.loc[games["Result"] == res, "plies"].clip(upper=240), bins=edges,
            histtype="step", lw=2, color=color, label=label)
ax.axvline(cfg.min_draw_plies, color=AXIS, lw=1)
ax.annotate(f"draws shorter than {cfg.min_draw_plies} plies\nare mostly agreed, not played out",
            (cfg.min_draw_plies, ax.get_ylim()[1] * 0.92), xytext=(8, 0), textcoords="offset points",
            color=INK2, fontsize=9, va="top")
ax.set_title("Game length by result")
ax.set_xlabel("plies (half-moves)")
ax.set_ylabel("games")
ax.legend(loc="upper right")

ax = axes[1]
top = games["Opening"].value_counts().head(10)[::-1]
ax.barh(top.index, top.values, color=SERIES[0], height=0.6)
for y, v in enumerate(top.values):
    ax.annotate(f"{v}", (v, y), xytext=(3, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
ax.set_title("Most common openings")
ax.set_xlabel("games")
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

The short spike of **draws** comes from games agreed before move 20: a
"grandmaster draw" reflects a decision by the players, not the position. Those
labels would teach the model that sharp opening positions are dead equal, so they
are removed in the cleaning step. Very short decisive games are rare, and are
dropped under the same rule (usually forfeits or opening accidents).

## 2. Cleaning: replay every game through the engine

Parsing the text is not enough. Every move is replayed through the C++ engine,
which validates it against the rules and yields the exact position after each
ply. `replay_games` runs multithreaded in C++ and returns all positions of all
games as one `(P, 72)` uint8 array, one compact position per row.

In [ ]:
t0 = time.time()
rep = ce.replay_games(games["movetext"].tolist())
print(f"replayed {len(games):,} games → {len(rep['boards']):,} positions in {time.time() - t0:.2f}s")

games["plies"] = rep["n_plies"]
games["replay_error"] = rep["error"]
games["result_token"] = rep["result_token"]

checks = pd.Series({
    "illegal or unreadable move": (games["replay_error"] != "").sum(),
    "movetext result ≠ Result tag": (games["result_token"] != games["Result"]).sum(),
    "duplicate move sequences": games.duplicated("movetext").sum(),
    f"shorter than {cfg.min_plies} plies": (games["plies"] < cfg.min_plies).sum(),
    f"draw shorter than {cfg.min_draw_plies} plies": ((games["Result"] == "1/2-1/2")
                                                     & (games["plies"] < cfg.min_draw_plies)).sum(),
})
checks.to_frame("games flagged")

In [ ]:
keep = ((games["replay_error"] == "")
        & (games["result_token"] == games["Result"])
        & ~games.duplicated("movetext")
        & (games["plies"] >= cfg.min_plies)
        & ~((games["Result"] == "1/2-1/2") & (games["plies"] < cfg.min_draw_plies)))
clean = games[keep].copy()
print(f"kept {len(clean):,} of {len(games):,} games ({keep.mean():.1%})")
clean["Result"].value_counts(normalize=True).reindex(["1-0", "1/2-1/2", "0-1"]).round(3).to_frame("share")

## 3. From games to labelled positions

**Unit of prediction.** A training example is a *position*, labelled with the final
result of the game it occurred in. The label is expressed from the point of view of
the side to move: **win**, **draw** or **loss**. A position after White's move 20
in a game Black won is a *win* example, since Black is to move.

This is noisy supervision. An early position doesn't determine the result, and
humans make mistakes later. With many games, though, the model learns the
*expected* outcome of each kind of position. That is exactly what a player needs
to compare moves.

In [ ]:
pos_game = rep["game"]
sel = np.isin(pos_game, clean.index.to_numpy())
boards = rep["boards"][sel]                      # (P, 72) uint8 — compact positions
pos_game = pos_game[sel]
pos_ply = rep["ply"][sel]

side = ce.board_fields(boards)["side"]           # 0 = white to move, 1 = black
white_score = games["white_score"].to_numpy()[pos_game]
stm_score = np.where(side == ce.WHITE, white_score, 1.0 - white_score)
labels = np.select([stm_score == 1.0, stm_score == 0.5], [0, 1], 2).astype(np.int64)   # 0 win, 1 draw, 2 loss
CLASSES = ["win", "draw", "loss"]

print(f"{len(boards):,} positions from {len(clean):,} games "
      f"({len(boards) / len(clean):.1f} per game); {boards.nbytes / 1e6:.0f} MB as compact boards")
pd.Series(labels).value_counts(normalize=True).sort_index().set_axis(CLASSES).round(3).to_frame("share (side to move)")

### Encoding a position for the network

The C++ engine turns a 72-byte position into **19 planes of 8×8**
(`core/encode.hpp`, one definition for the CPU, the CUDA kernels and the PyTorch
fallback):

| planes | feature | encoding |
|---|---|---|
| 0–5 | our pawns, knights, bishops, rooks, queens, king | one-hot per square |
| 6–11 | their pieces, same order | one-hot per square |
| 12–15 | castling rights: our king-side, our queen-side, their king-side, their queen-side | binary, constant plane |
| 16 | en-passant target square | one-hot |
| 17 | half-move clock (50-move rule) ÷ 100 | numeric, scaled to [0, 1] |
| 18 | all ones | lets zero-padded convolutions find the board edge |

The design choices follow standard data-science practice:

* **Categorical data is one-hot encoded.** A square's content is a category (empty
  or one of 12 pieces). A single integer "piece code" would impose a false order,
  as if a king were "more" than a queen. Twelve binary planes don't, and "empty" is
  the all-zero case (drop-one encoding).
* **Castling rights are four independent yes/no facts,** so they get four binary
  planes. They matter: castling is part of the state even when the pieces look
  identical.
* **The one numeric feature is scaled** to the same [0, 1] range as everything else.
* **The board is seen from the side to move.** When Black is to move, the board is
  flipped and colours swapped. The rules are symmetric under this transform, so the
  model learns one function instead of two, and the labels are win/draw/loss for
  "us". Converting back to White/Black probabilities is a relabelling.

In [ ]:
example = ce.Game()
for san in "e4 c5 Nf3 d6 d4 cxd4 Nxd4 Nf6 Nc3 a6 Be3 e5 Nb3 Be6 f3 Be7 Qd2 O-O O-O-O".split():
    example.push_san(san)
print(example.fen(), "\n")
print(example)
x = example.encode()                               # the per-turn hook: (19, 8, 8) float32
print("encoded:", x.shape, x.dtype, "— Black to move, so seen from Black's side")

PLANE_TITLES = ["our P", "our N", "our B", "our R", "our Q", "our K",
                "their P", "their N", "their B", "their R", "their Q", "their K",
                "our O-O", "our O-O-O", "their O-O", "their O-O-O", "en passant", "50-move clock", "ones"]
fig, axes = plt.subplots(2, 10, figsize=(13, 3.3))
for p, ax in enumerate(axes.flat):
    if p >= ce.NUM_PLANES:
        ax.axis("off")
        continue
    ax.imshow(x[p], cmap=BLUES, vmin=0, vmax=1)
    ax.set_xticks(np.arange(-0.5, 8, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 8, 1), minor=True)
    ax.grid(which="minor", color=GRID, linewidth=0.5)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="both", length=0, labelbottom=False, labelleft=False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(AXIS)
    ax.set_title(PLANE_TITLES[p], fontsize=8, color=INK2, loc="center", fontweight="normal")
fig.suptitle("The 19 input planes of the example position (dark = 1)", x=0.01, ha="left",
             color=INK, fontsize=11.5, fontweight="bold")
fig.tight_layout()
plt.show()

Black is to move, so the planes show the board from Black's side. Black's pieces
are "ours" and sit at the bottom (Black's king on g8 appears where g1 would be),
and White's pieces, with the king castled to c1, are "theirs" at the top. The
printed diagram above is White's view. Both sides have castled, so all four
castling planes are empty. The half-move clock plane is faint: four quiet moves
since White's last pawn move.

## 4. Train / validation / test split — by game

All positions of one game share a label and are highly correlated. Splitting
positions at random would put near-copies of the same game in both training and
test data (**leakage**), and the test score would reward memorising games. The
split is therefore made over **games**, stratified by result so that each part has
the same outcome mix. The test set is locked away until the final evaluation.

In [ ]:
game_ids = clean.index.to_numpy()
train_g, hold_g = train_test_split(game_ids, test_size=cfg.val_frac + cfg.test_frac,
                                   stratify=clean["Result"], random_state=cfg.seed)
val_g, test_g = train_test_split(hold_g, test_size=cfg.test_frac / (cfg.val_frac + cfg.test_frac),
                                 stratify=clean.loc[hold_g, "Result"], random_state=cfg.seed)
split = np.select([np.isin(pos_game, train_g), np.isin(pos_game, val_g)], [0, 1], 2)
SPLITS = ["train", "val", "test"]

assert not (set(train_g) & set(val_g) or set(train_g) & set(test_g) or set(val_g) & set(test_g))
summary = pd.DataFrame({
    "games": [len(train_g), len(val_g), len(test_g)],
    "positions": [(split == s).sum() for s in range(3)],
    **{f"{c} %": [100 * (labels[split == s] == k).mean() for s in range(3)] for k, c in enumerate(CLASSES)},
}, index=SPLITS)
summary.round(1)

Identical positions *can* still occur in different games, mostly well-known opening
positions. That isn't leakage: the same input legitimately appears at play time
too, and its label is a different game's result. Measure it anyway, so the
evaluation can be read correctly.

In [ ]:
keys = boards.view(np.dtype((np.void, ce.BOARD_BYTES))).ravel()
train_keys = set(keys[split == 0].tolist())
seen = np.array([k in train_keys for k in keys[split == 2].tolist()])
test_plies = pos_ply[split == 2]
print(f"test positions that also occur in training games: {seen.mean():.1%}")
print(f"  of those, {np.mean(test_plies[seen] < 20):.0%} are in the first 20 plies (shared opening theory)")

## 5. The data pipeline — encoding on the GPU

Materialising the float32 input tensors for every position would take
`P × 19 × 64 × 4` bytes, several gigabytes. Instead, the compact boards (tens of
megabytes) go to the device once. Each mini-batch is then:

1. a random slice of board indices,
2. **encoded by the engine's CUDA kernel** (`gpu.encode`: one GPU thread per output
   value, so writes are coalesced),
3. optionally **mirrored left↔right**. This data augmentation is valid only when
   neither side can castle: the rules are symmetric under a file mirror except for
   castling (the king starts on the e-file).

On a CPU the same code calls the multithreaded C++ encoder instead.

In [ ]:
class PositionLoader:
    # Mini-batches (x, y) produced on `device` from compact 72-byte boards.

    def __init__(self, boards, labels, batch_size, device, shuffle=False, augment=False,
                 samples=None, seed=0):
        self.boards = torch.from_numpy(np.ascontiguousarray(boards)).to(device)
        self.labels = torch.from_numpy(labels).to(device)
        self.batch_size, self.shuffle, self.augment, self.samples = batch_size, shuffle, augment, samples
        self.gen = torch.Generator(device=device).manual_seed(seed)

    def __len__(self):
        n = min(self.samples or len(self.labels), len(self.labels))
        return math.ceil(n / self.batch_size)

    def __iter__(self):
        n = len(self.labels)
        dev = self.labels.device
        order = (torch.randperm(n, device=dev, generator=self.gen) if self.shuffle
                 else torch.arange(n, device=dev))[: self.samples or n]
        for i in range(0, len(order), self.batch_size):
            idx = order[i : i + self.batch_size]
            b = self.boards[idx]
            x = gpu.encode(b)                                   # CUDA kernel (or C++ on CPU)
            if self.augment:
                can_mirror = b[:, 65] == 0                      # no castling rights left
                flip = can_mirror & (torch.rand(len(idx), device=dev, generator=self.gen) < 0.5)
                x[flip] = x[flip].flip(-1)
            yield x, self.labels[idx]


loaders = {
    "train": PositionLoader(boards[split == 0], labels[split == 0], cfg.batch_size, DEVICE, shuffle=True,
                            augment=cfg.mirror_augment, samples=cfg.samples_per_epoch, seed=cfg.seed),
    "val": PositionLoader(boards[split == 1], labels[split == 1], 4 * cfg.batch_size, DEVICE),
    "test": PositionLoader(boards[split == 2], labels[split == 2], 4 * cfg.batch_size, DEVICE),
}
xb, yb = next(iter(loaders["train"]))
print("batch:", tuple(xb.shape), xb.dtype, "on", xb.device, "| labels:", tuple(yb.shape))

## 6. Baselines first

A model is only interesting if it beats simple alternatives. Three of them bracket
what the CNN has to add:

* **Prior** — always predict the training set's class frequencies. Knows nothing
  about the position.
* **Material** — a logistic regression on piece counts: ours and theirs, 10
  features, standardised. This is the "count the material" rule of thumb, learned
  from data. The web app's current *Collector* and *Lookahead* bots also think in
  material.
* **Linear on planes** — a logistic regression on all 1,216 input values (a
  learned piece-square table), trained with the same loop as the CNN. Knows where
  pieces stand, but no interactions between them.

The main metric is **log loss** (cross-entropy). It is a proper scoring rule for
probabilities, and probabilities are what move selection consumes. Accuracy is
reported too, but "predict the draw" gets a lot of accuracy for free.

In [ ]:
def material_features(b):
    # Counts of each piece type (no kings), ours then theirs, from the side to move's view.
    f = ce.board_fields(b)
    sq, white = f["sq"], f["side"] == ce.WHITE
    w = np.stack([(sq == t).sum(1) for t in range(1, 6)], 1)          # white P N B R Q
    bl = np.stack([(sq == 8 + t).sum(1) for t in range(1, 6)], 1)     # black P N B R Q
    return np.where(white[:, None], np.hstack([w, bl]), np.hstack([bl, w])).astype(np.float32)


def evaluate_probs(probs, y):
    onehot = np.eye(3)[y]
    return {"log loss": log_loss(y, probs, labels=[0, 1, 2]),
            "Brier": np.mean(np.sum((probs - onehot) ** 2, 1)),
            "accuracy": accuracy_score(y, probs.argmax(1)),
            "macro F1": f1_score(y, probs.argmax(1), average="macro")}


y_train, y_val, y_test = (labels[split == s] for s in range(3))
prior = np.bincount(y_train, minlength=3) / len(y_train)
results = {"prior": evaluate_probs(np.tile(prior, (len(y_val), 1)), y_val)}

material_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
material_lr.fit(material_features(boards[split == 0]), y_train)
results["material LR"] = evaluate_probs(material_lr.predict_proba(material_features(boards[split == 1])), y_val)
pd.DataFrame(results).T.round(4)

## 7. The model: a residual CNN with a win/draw/loss head

A chessboard is an 8×8 grid, and much of chess knowledge is local patterns: pawn
chains, a knight's reach, pieces around the king. Convolutions learn such
patterns once and apply them everywhere on the board. Residual blocks (He et al.,
2016) keep a deeper stack trainable. This is the same family of network as
AlphaZero's and Leela Chess Zero's value heads, at a scale that suits 10,000 games.

```
input 19×8×8 → conv3×3 → [residual block] × B → conv1×1 → dense → dropout → 3 logits
```

The output is three logits, turned into probabilities by a softmax and trained with
cross-entropy on win/draw/loss. The model's *expected score* for the side to move
is `P(win) + ½·P(draw)`.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        return F.relu(x + self.bn2(self.conv2(y)))


class ValueNet(nn.Module):
    # (N, 19, 8, 8) side-to-move planes -> (N, 3) logits over [win, draw, loss].

    def __init__(self, planes=19, channels=64, blocks=6, head_channels=32, hidden=128, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(planes, channels, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(channels), nn.ReLU(inplace=True))
        self.tower = nn.Sequential(*[ResidualBlock(channels) for _ in range(blocks)])
        self.head = nn.Sequential(
            nn.Conv2d(channels, head_channels, 1, bias=False), nn.BatchNorm2d(head_channels), nn.ReLU(inplace=True),
            nn.Flatten(), nn.Linear(head_channels * 64, hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(hidden, 3))

    def forward(self, x):
        return self.head(self.tower(self.stem(x)))


def make_model(c=cfg):
    return ValueNet(ce.NUM_PLANES, c.channels, c.blocks, c.head_channels, c.hidden, c.dropout)


model = make_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ValueNet: {cfg.channels} channels × {cfg.blocks} residual blocks, {n_params:,} parameters")
print("output for one batch:", tuple(model(xb[:4]).shape))

### Training loop

* **AdamW** with decoupled weight decay (L2 regularisation), and a **one-cycle**
  learning-rate schedule: warm-up, then cosine decay.
* **Mixed precision** on GPU: bfloat16 where supported, else float16 with loss
  scaling. Gradients are clipped at norm 1.
* **Early stopping** on validation log loss. The best weights are kept, not the last.
* Losses are summed on the device and read once per epoch, so the GPU isn't
  stalled every step.

In [ ]:
AMP_DTYPE = (torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
             else torch.float16 if DEVICE.type == "cuda" else None)
torch.backends.cudnn.benchmark = True


@torch.no_grad()
def predict(model, loader):
    # Softmax probabilities (numpy) for every position of a loader, in order.
    model.eval()
    out = []
    for x, _ in loader:
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            out.append(torch.softmax(model(x).float(), 1))
    return torch.cat(out).cpu().numpy()


def train(model, loaders, epochs, lr, weight_decay, patience, label=""):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(loaders["train"]),
                                                pct_start=0.15)
    scaler = torch.amp.GradScaler(enabled=AMP_DTYPE == torch.float16)
    best = (float("inf"), None, -1)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        total, seen = torch.zeros((), device=DEVICE), 0
        for x, y in loaders["train"]:
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
                loss = F.cross_entropy(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            total += loss.detach().float() * len(y)
            seen += len(y)
        train_loss = total.item() / seen
        val = evaluate_probs(predict(model, loaders["val"]), y_val)
        history.append({"epoch": epoch, "train log loss": train_loss, "val log loss": val["log loss"],
                        "val accuracy": val["accuracy"], "positions/s": seen / (time.time() - t0)})
        improved = val["log loss"] < best[0] - 1e-4
        if improved:
            best = (val["log loss"], copy.deepcopy(model.state_dict()), epoch)
        print(f"{label}epoch {epoch:2d}  train {train_loss:.4f}  val {val['log loss']:.4f}  "
              f"acc {val['accuracy']:.3f}  {seen / (time.time() - t0):8,.0f} pos/s {'*' if improved else ''}")
        if epoch - best[2] >= patience:
            print(f"{label}early stop: no improvement for {patience} epochs")
            break
    model.load_state_dict(best[1])
    return pd.DataFrame(history)

First the linear baseline, trained with this same loop, so it gets exactly the same
data, encoding and optimiser.

In [ ]:
seed_everything(cfg.seed)
linear = nn.Sequential(nn.Flatten(), nn.Linear(ce.NUM_PLANES * 64, 3)).to(DEVICE)
_ = train(linear, loaders, epochs=min(4, cfg.epochs), lr=5e-3, weight_decay=1e-4, patience=2, label="linear ")
results["linear planes"] = evaluate_probs(predict(linear, loaders["val"]), y_val)

Now the CNN:

In [ ]:
seed_everything(cfg.seed)
model = make_model().to(DEVICE)
t0 = time.time()
history = train(model, loaders, cfg.epochs, cfg.lr, cfg.weight_decay, cfg.patience)
print(f"training took {(time.time() - t0) / 60:.1f} min")
results["CNN"] = evaluate_probs(predict(model, loaders["val"]), y_val)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
ax = axes[0]
ax.plot(history["epoch"], history["train log loss"], marker="o", ms=4, label="train")
ax.plot(history["epoch"], history["val log loss"], marker="o", ms=4, label="validation")
best_epoch = history.loc[history["val log loss"].idxmin()]
ax.annotate(f"best {best_epoch['val log loss']:.4f}", (best_epoch["epoch"], best_epoch["val log loss"]),
            xytext=(0, -16), textcoords="offset points", ha="center", color=INK2, fontsize=9)
ax.set_title("Learning curves")
ax.set_xlabel("epoch")
ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
ax.set_ylabel("log loss")
ax.legend()
ax = axes[1]
ax.plot(history["epoch"], history["val accuracy"], marker="o", ms=4, color=SERIES[1])
ax.set_title("Validation accuracy")
ax.set_xlabel("epoch")
ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0, decimals=1))
fig.tight_layout()
plt.show()
pd.DataFrame(results).T.round(4)

The training loss keeps falling while the validation loss flattens. That is the
usual sign that the model starts to memorise individual games. Early stopping
keeps the weights from the best validation epoch. The CNN should beat all three
baselines on validation log loss. If it doesn't, the extra capacity isn't paying
for itself.

## 8. Final evaluation on the test set

The test games have not influenced any choice so far, so this is the honest
estimate of how the model does on new games.

In [ ]:
test_probs = {
    "prior": np.tile(prior, (len(y_test), 1)),
    "material LR": material_lr.predict_proba(material_features(boards[split == 2])),
    "linear planes": predict(linear, loaders["test"]),
    "CNN": predict(model, loaders["test"]),
}
test_table = pd.DataFrame({k: evaluate_probs(p, y_test) for k, p in test_probs.items()}).T
test_table.round(4)

In [ ]:
p_cnn = test_probs["CNN"]
cm = confusion_matrix(y_test, p_cnn.argmax(1), labels=[0, 1, 2])
expected = p_cnn[:, 0] + 0.5 * p_cnn[:, 1]                       # predicted expected score
actual = np.array([1.0, 0.5, 0.0])[y_test]
bins = np.linspace(0, 1, 11)
which = np.clip(np.digitize(expected, bins) - 1, 0, 9)
calib = pd.DataFrame({"pred": expected, "actual": actual, "bin": which}).groupby("bin").agg(
    pred=("pred", "mean"), actual=("actual", "mean"), n=("pred", "size"))
ece = np.sum(calib["n"] * (calib["pred"] - calib["actual"]).abs()) / len(expected)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
rowp = cm / cm.sum(1, keepdims=True)
ax.imshow(rowp, cmap=BLUES, vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{rowp[i, j]:.0%}\n{cm[i, j]:,}", ha="center", va="center", fontsize=9,
                color="white" if rowp[i, j] > 0.55 else INK)
ax.set_xticks(range(3), CLASSES)
ax.set_yticks(range(3), CLASSES)
ax.set_xlabel("predicted (side to move)")
ax.set_ylabel("actual result")
ax.grid(False)
ax.set_title("Confusion matrix (row-normalised)")
ax = axes[1]
ax.plot([0, 1], [0, 1], color=AXIS, lw=1, label="perfect calibration")
ax.plot(calib["pred"], calib["actual"], marker="o", ms=5, color=SERIES[0], label="CNN")
ax.set_title(f"Calibration of the expected score (ECE {ece:.3f})")
ax.set_xlabel("predicted P(win) + ½ P(draw)")
ax.set_ylabel("observed score")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

**Calibration** asks whether an expected score of 0.7 turns into 0.7 points on
average. Points near the diagonal mean the probabilities can be taken at face
value, which move selection relies on. The confusion matrix shows the draw class is
the hardest: a draw is the "undecided" outcome, and many drawn positions look like
small edges for one side.

In [ ]:
phase_edges = [0, 10, 20, 30, 40, 50, 60, 80, 100, 1000]
phase = pd.cut(pos_ply[split == 2], phase_edges, right=False, labels=[
    f"{a}–{b - 1}" if b < 1000 else f"{a}+" for a, b in zip(phase_edges[:-1], phase_edges[1:])])
by_phase = pd.DataFrame({
    name: pd.Series([log_loss(y_test[phase == c], p[phase == c], labels=[0, 1, 2]) for c in phase.categories],
                    index=phase.categories)
    for name, p in test_probs.items()})

fig, ax = plt.subplots(figsize=(10, 3.6))
for name, color in zip(["prior", "material LR", "linear planes", "CNN"], [MUTED, SERIES[1], SERIES[2], SERIES[0]]):
    ax.plot(range(len(by_phase)), by_phase[name], marker="o", ms=4, color=color, label=name,
            lw=2.5 if name == "CNN" else 1.8)
ax.set_xticks(range(len(by_phase)), by_phase.index)
ax.set_title("Test log loss by game phase (lower is better)")
ax.set_xlabel("ply of the position")
ax.set_ylabel("log loss")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.tight_layout()
plt.show()

Early positions are nearly uninformative: every model sits near the prior, because
the opening rarely decides a game between masters. As the game goes on the board
says more and more about the result. The gap between the CNN and material counting
is the positional understanding the convolutional layers add.

### Error analysis: the most confident mistakes

Look at examples, not only aggregates. These are test positions where the model
put the most probability on an outcome that did not happen.

In [ ]:
test_idx = np.flatnonzero(split == 2)
wrong_conf = p_cnn.max(1) * (p_cnn.argmax(1) != y_test)
rows, seen_games = [], set()
for k in np.argsort(-wrong_conf):
    i = test_idx[k]
    if wrong_conf[k] == 0 or len(rows) == 6:
        break
    if pos_game[i] in seen_games:          # one position per game
        continue
    seen_games.add(pos_game[i])
    g = games.loc[pos_game[i]]
    rows.append({"game": f"{g['White']} – {g['Black']}", "result": g["Result"], "ply": pos_ply[i],
                 "predicted (side to move)": CLASSES[p_cnn[k].argmax()], "confidence": p_cnn[k].max(),
                 "actual": CLASSES[y_test[k]], "FEN": ce.board_to_fen(boards[i])})
pd.DataFrame(rows)

Typically these are positions with a large material or positional edge where the
game still went the other way: a later blunder, a swindle, or a fortress. No model
of the position alone can predict those. This label noise is the floor under the
test loss.

## 9. From the network's view back to White and Black

The network speaks from the side to move's perspective. To answer "who is going to
win this game?", relabel: if White is to move, *P(White wins)* = *P(win)*;
otherwise it is *P(loss)*.

In [ ]:
evaluator = player.TorchEvaluator(model, DEVICE)


def white_black(game):
    p = evaluator(game.encode()[None])[0]
    w, d, l = (p if game.turn == ce.WHITE else p[::-1])
    return {"P(White wins)": w, "P(draw)": d, "P(Black wins)": l}


positions = {
    "start position": ce.Game(),
    "the example position (Sicilian, English Attack)": example,
    "White up a queen (after a blunder)": ce.Game("r1b1kbnr/pppp1ppp/2n5/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 0 4"),
    "K+R vs K endgame, White to move": ce.Game("8/8/4k3/8/8/2K5/6R1/8 w - - 0 60"),
    "White to move, a knight down": ce.Game("rnbqkb1r/pppp1ppp/5n2/4p3/4P3/8/PPPP1PPP/RNBQKB1R w KQkq - 0 3"),
}
pd.DataFrame({name: white_black(g) for name, g in positions.items()}).T.round(3)

## 10. Inference: playing moves

The model plays by **one-ply lookahead**. For every legal move, the engine builds
the resulting position (`Game.children()`), and the network evaluates it, all
positions in one batch. The move with the best expected score for the mover is
played. After the move the *opponent* is to move, so the network's
"loss" becomes our "win":

`score(move) = P(opponent loses) + ½·P(draw)`

Positions the rules already decide skip the network: a mating move scores 1, and
stalemate, insufficient material, the 50-move rule and threefold repetition score
½. A mate in one is always played. A confident network can score many winning
moves at exactly 1.0, but only the mate ends the game.

This lives in `chess_engine.player`, so the deployed bot and this notebook share one
implementation:

In [ ]:
print(inspect.getsource(player.score_moves))
print(inspect.getsource(player.choose_move))

In [ ]:
def show_moves(game, top=6):
    scored = sorted(player.score_moves(game, evaluator), key=lambda s: -s.score)[:top]
    return pd.DataFrame([{"move": s.san, "expected score": s.score,
                          "W/D/L for the mover": "decided: " + s.terminal if s.terminal
                          else "{:.2f} / {:.2f} / {:.2f}".format(*s.wdl)} for s in scored])


print("Back-rank mate available:")
display(show_moves(ce.Game("6k1/5ppp/8/8/8/8/5PPP/3R2K1 w - - 0 1"), 4))
print("A hanging queen:")
display(show_moves(ce.Game("rnb1kbnr/pppp1ppp/8/4p1q1/4P3/3P4/PPP2PPP/RNBQKBNR w KQkq - 1 3"), 4))
print("The example position, Black to move:")
display(show_moves(example))

### A game, turn by turn

The single-game interface as the web app would use it. At every turn the engine
hands the current state to the network, `player.choose_move` asks for a decision,
and the move is played. Here the network (White) meets a simple *material-greedy*
opponent, similar to the web app's *Collector* bot.

In [ ]:
def greedy_material_move(game, rng):
    # The web app's "Collector" idea: take mate, else maximise material; ties at random.
    values = {"p": 1, "n": 3, "b": 3, "r": 5, "q": 9, "k": 0}
    best, best_score = [], -math.inf
    for move in game.legal_moves():
        game.push(move)
        if game.is_checkmate():
            score = 1e9
        elif game.is_game_over():
            score = 0.0
        else:
            board = str(game).split()
            mine = sum(values[c.lower()] for c in board if c != "." and (c.isupper() == (game.turn == ce.BLACK)))
            theirs = sum(values[c.lower()] for c in board if c != "." and (c.isupper() == (game.turn == ce.WHITE)))
            score = mine - theirs
        game.undo()
        if score > best_score:
            best, best_score = [move], score
        elif score == best_score:
            best.append(move)
    return rng.choice(best)


rng = random.Random(cfg.seed)
game = ce.Game()
white_expected = []                         # the network's view of the game, turn by turn
while not game.is_game_over() and game.ply < 200:
    w, d, l = evaluator(game.encode()[None])[0]          # send the current state to the network
    white_expected.append(w + 0.5 * d if game.turn == ce.WHITE else l + 0.5 * d)
    if game.turn == ce.WHITE:
        move = player.choose_move(game, evaluator, rng=rng).move
    else:
        move = greedy_material_move(game, rng)
    game.push(move)
print(game.pgn())
outcome = ("checkmate" if game.is_checkmate() else game.draw_reason().name.lower() if game.is_game_over()
           else "move limit")
print("\nresult:", game.result().name, f"({outcome})")

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.axhline(0.5, color=AXIS, lw=1)
ax.plot(np.arange(len(white_expected)), white_expected, color=SERIES[0])
ax.set_ylim(0, 1)
ax.set_title("The network's expected score for White, every turn of the demo game")
ax.set_xlabel("ply")
ax.set_ylabel("P(W) + ½ P(D)")
fig.tight_layout()
plt.show()

### Many games at once, on the GPU

One game at a time is right for serving a human. To *measure* the model we play
hundreds of games **in parallel**. Each ply, all positions of all running games
are expanded into their children by the CUDA kernels (`gpu.expand`), encoded
(`gpu.encode`) and evaluated in one forward pass. Each game's best child is then
picked with a segmented arg-max (`player.select_moves`), with no Python loop over
games.

Opponents: **random** (a sanity check) and **greedy material** (take mate, else
the most material; ties at random). Each game starts with a few random plies for
variety. Colours alternate. Games still running at the move limit count as draws.
The batched path has no threefold-repetition detection, since compact boards carry
no history.

In [ ]:
PIECE_VALUES = torch.tensor([1.0, 3.0, 3.0, 5.0, 9.0, 0.0], device=DEVICE)


def children_choice(boards, key, parent, offsets):
    # For each board, the index of its child with the largest key (first on ties).
    best = torch.full((len(boards),), -math.inf, device=key.device).scatter_reduce(0, parent, key, "amax")
    idx = torch.arange(len(key), device=key.device)
    return torch.full((len(boards),), len(key), device=key.device).scatter_reduce(
        0, parent, torch.where(key == best[parent], idx, len(key)), "amin")


def random_policy(boards, gen):
    children, _, _, offsets = gpu.expand(boards)
    counts = offsets[1:] - offsets[:-1]
    pick = offsets[:-1] + (torch.rand(len(boards), device=boards.device, generator=gen) * counts).long()
    return children[pick]


def greedy_material_policy(boards, gen):
    children, _, parent, offsets = gpu.expand(boards)
    _, status = gpu.count_legal(children)
    pieces = gpu.encode(children)[:, :12].sum((2, 3))        # child's view: planes 6-11 are the mover's
    material = pieces[:, 6:12] @ PIECE_VALUES - pieces[:, 0:6] @ PIECE_VALUES
    st = status.long()
    key = torch.where(st == int(ce.Status.CHECKMATE), torch.full_like(material, 1e9),
                      torch.where(st != int(ce.Status.ONGOING), torch.zeros_like(material), material))
    key = key + 0.01 * torch.rand(len(key), device=key.device, generator=gen)   # random tie-break
    return children[children_choice(boards, key, parent, offsets)]


def model_policy(boards, gen):
    return player.select_moves(boards, model)["child"]


@torch.no_grad()
def play_match(policy, opponent, n_games, max_plies, random_plies, seed):
    model.eval()
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    boards = torch.from_numpy(ce.startpos()).to(DEVICE).repeat(n_games, 1)
    policy_white = torch.arange(n_games, device=DEVICE) % 2 == 0
    white_score = torch.full((n_games,), 0.5, device=DEVICE)
    done = torch.zeros(n_games, dtype=torch.bool, device=DEVICE)
    for ply in range(max_plies):
        active = ~done
        if not active.any():
            break
        white_to_move = boards[:, 64] == 0
        for fn, mask in ((policy, active & (white_to_move == policy_white)),
                         (opponent, active & (white_to_move != policy_white))):
            if mask.any():
                fn = random_policy if ply < random_plies else fn
                boards[mask] = fn(boards[mask], gen)
        _, status = gpu.count_legal(boards)
        ended = active & (status != int(ce.Status.ONGOING))
        mated = ended & (status == int(ce.Status.CHECKMATE))
        white_score[mated] = (boards[mated, 64] == 1).float()   # side to move is mated
        done |= ended
    score = torch.where(policy_white, white_score, 1 - white_score).cpu().numpy()
    return {"win": np.mean(score == 1), "draw": np.mean(score == 0.5), "loss": np.mean(score == 0),
            "score": score.mean(), "unfinished": (~done).float().mean().item(), "games": n_games}


def elo_diff(score, n):
    s = np.clip(score, 1e-3, 1 - 1e-3)
    se = np.sqrt(s * (1 - s) / n)
    to_elo = lambda p: -400 * np.log10(1 / np.clip(p, 1e-3, 1 - 1e-3) - 1)
    return to_elo(s), to_elo(s - 1.96 * se), to_elo(s + 1.96 * se)


matches = {}
for name, opp in [("random mover", random_policy), ("greedy material", greedy_material_policy)]:
    t0 = time.time()
    matches[name] = play_match(model_policy, opp, cfg.match_games, cfg.match_max_plies, cfg.opening_random_plies,
                               seed=cfg.seed)
    elo, lo, hi = elo_diff(matches[name]["score"], cfg.match_games)
    matches[name].update({"Elo difference": elo, "95% CI": f"{lo:+.0f} … {hi:+.0f}", "seconds": time.time() - t0})
pd.DataFrame({name: {"games": m["games"], "win %": 100 * m["win"], "draw %": 100 * m["draw"],
                     "loss %": 100 * m["loss"], "score %": 100 * m["score"],
                     "unfinished %": 100 * m["unfinished"], "Elo difference": round(m["Elo difference"]),
                     "95% CI": m["95% CI"], "seconds": round(m["seconds"], 1)}
              for name, m in matches.items()}).T

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.6))
names = list(matches)
left = np.zeros(len(names))
for key, color, label in [("win", WIN, "network wins"), ("draw", DRAW, "draw"), ("loss", LOSS, "network loses")]:
    vals = np.array([matches[n][key] for n in names]) * 100
    ax.barh(names, vals, left=left, color=color, height=0.5, label=label, edgecolor=SURFACE, linewidth=2)
    for y, (l, v) in enumerate(zip(left, vals)):
        if v >= 7:
            ax.text(l + v / 2, y, f"{v:.0f}%", ha="center", va="center", color="white", fontsize=9)
    left += vals
ax.set_xlim(0, 100)
ax.set_xlabel("% of games")
ax.set_title(f"The network against simple opponents ({cfg.match_games} games each, colours alternating)")
ax.grid(axis="y", visible=False)
ax.invert_yaxis()
fig.legend(*ax.get_legend_handles_labels(), ncol=3, loc="lower center", frameon=False)
fig.tight_layout(rect=(0, 0.14, 1, 1))
plt.show()

Beating a random mover is a sanity check. The greedy material player is the real
yardstick: it never misses a capture or a mate in one, so to beat it the network
has to prefer *positions* that lead to gains, not just immediate captures. Draws in
these matches are often move-limit adjudications. One-ply lookahead finds a mate
only when it is one move away, so converting a won position into mate is this
player's weakest skill.

## 11. Export for deployment

Three artefacts, plus a model card:

* **`value_net.onnx`** — the network as a portable graph with a dynamic batch
  dimension. It runs with ONNX Runtime in Python, Node, or the **browser**
  (`onnxruntime-web`, next to the WASM chess engine). This is the deployment format.
* **`value_net.pt2`** — a `torch.export` program. It loads in PyTorch without this
  notebook's class definitions.
* **`value_net_checkpoint.pt`** — weights, configuration and metrics, for
  fine-tuning.
* **`model_card.json`** — the input contract (plane order, side-to-move
  perspective), output order, training data and test metrics.

Each exported file is loaded back and checked against the in-memory model.

In [ ]:
import logging
logging.getLogger("torch.onnx").setLevel(logging.ERROR)     # exporter chatter (e.g. torchvision ops)

EXPORT_DIR = ROOT / "ai" / "models"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
cpu_model = copy.deepcopy(model).cpu().eval()
sample = torch.from_numpy(ce.encode_boards(boards[split == 2][:64]))

torch.save({"state_dict": cpu_model.state_dict(), "config": asdict(cfg), "test_metrics": test_table.loc["CNN"].to_dict()},
           EXPORT_DIR / "value_net_checkpoint.pt")

batch = torch.export.Dim("batch", min=1, max=65536)
exported = torch.export.export(cpu_model, (sample,), dynamic_shapes=({0: batch},))
torch.export.save(exported, EXPORT_DIR / "value_net.pt2")

onnx_path = EXPORT_DIR / "value_net.onnx"
try:   # the torch.export-based exporter (PyTorch >= 2.5)
    torch.onnx.export(cpu_model, (sample,), onnx_path, input_names=["planes"], output_names=["wdl_logits"],
                      dynamic_shapes=({0: batch},), dynamo=True, external_data=False, verbose=False)
except Exception as err:  # older PyTorch: the TorchScript-based exporter
    print("dynamo ONNX export unavailable, using the legacy exporter:", type(err).__name__)
    torch.onnx.export(cpu_model, (sample,), onnx_path, input_names=["planes"], output_names=["wdl_logits"],
                      dynamic_axes={"planes": {0: "batch"}, "wdl_logits": {0: "batch"}}, opset_version=17,
                      dynamo=False)

with torch.no_grad():
    reference = torch.softmax(cpu_model(sample), 1).numpy()
checks = {"pt2": player.load_evaluator(str(EXPORT_DIR / "value_net.pt2"))(sample.numpy())}
try:
    checks["onnx"] = player.load_evaluator(str(onnx_path))(sample.numpy())
except ImportError:
    print("onnxruntime not installed: skipping the ONNX check (pip install onnxruntime)")
for name, probs in checks.items():
    print(f"{name}: max |Δp| vs PyTorch = {np.abs(probs - reference).max():.2e}")
    assert np.allclose(probs, reference, atol=1e-4)

In [ ]:
card = {
    "model": "PyChess value network (residual CNN)",
    "task": "P(win), P(draw), P(loss) for the side to move",
    "input": {"name": "planes", "shape": ["batch", ce.NUM_PLANES, 8, 8], "dtype": "float32",
              "planes": ce.PLANE_NAMES,
              "perspective": "side to move; board flipped vertically and colours swapped when black is to move",
              "square_order": "row-major, row 0 = the opponent's back rank",
              "reference": "core/encode.hpp (C++/CUDA), chess_engine.gpu.encode_torch (PyTorch)"},
    "output": {"name": "wdl_logits", "shape": ["batch", 3], "order": ["win", "draw", "loss"],
               "activation": "softmax",
               "move_selection": "play argmax over legal moves of P(opponent loses) + 0.5 * P(draw) "
                                 "in the position after the move; mate in one first"},
    "architecture": {k: getattr(cfg, k) for k in ("channels", "blocks", "head_channels", "hidden", "dropout")}
                    | {"parameters": n_params},
    "training": {"profile": PROFILE, "device": str(DEVICE), "epochs_run": int(history["epoch"].max()),
                 "best_epoch": int(history.loc[history["val log loss"].idxmin(), "epoch"]),
                 "games": {"train": len(train_g), "val": len(val_g), "test": len(test_g)},
                 "positions": {"train": int((split == 0).sum()), "val": int((split == 1).sum()),
                               "test": int((split == 2).sum())},
                 "data": manifest["source"] + ", " + " to ".join(manifest["date_range"])},
    "test_metrics": {name: {k: round(float(v), 4) for k, v in row.items()} for name, row in test_table.iterrows()},
    "matches": {name: {k: (round(float(v), 3) if isinstance(v, (float, np.floating)) else v) for k, v in m.items()}
                for name, m in matches.items()},
    "created": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
(EXPORT_DIR / "model_card.json").write_text(json.dumps(card, indent=2, default=str))
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:6.2f} MB")
if IN_COLAB:
    print("\nDownload the files from the Colab file browser (ai/models/), or:\n"
          "  from google.colab import files; files.download('/content/chess/ai/models/value_net.onnx')")

The exported model can be played against straight away in a terminal:

```sh
pip install . onnxruntime                  # from the repository root
python -m chess_engine.player ai/models/value_net.onnx --color white
```

For the web app, the path is `onnxruntime-web` next to the existing WASM engine.
The encoding in `core/encode.hpp` compiles to WASM with the rest of the engine,
so the browser can produce exactly the inputs the network was trained on.

## 12. Conclusions and next steps

**What we built.** A reproducible pipeline from raw PGN to a playing model:

* engine-validated data with documented cleaning;
* leakage-free splits;
* one-hot, side-to-move features, including castling rights;
* baselines;
* a residual CNN trained with the data on the GPU;
* calibration and error analysis;
* a move-selection policy shared between the notebook and deployment.

**Limitations to keep in mind.**

* *Label noise:* a game's result is a weak label for its early positions. Ratings,
  time trouble and later blunders are invisible to a board-only model.
* *Data volume:* 10,000 games (~0.8 M correlated positions) is small for chess.
  The learning curves show where memorisation starts.
* *One-ply lookahead:* the player sees mates in one, but not deeper tactics.
* *No history in the input:* repetition is handled by the rules layer, not the
  network.

**Ways to make it stronger, roughly in order of payoff.**

1. **More data.** `ai/data/download_games.py --target 50000` (or a lower Elo
   bar) needs no other change.
2. **Search.** A 2–3 ply minimax or MCTS on top of the network, using the
   batched GPU move generator already here.
3. **Better labels.** Engine evaluations (e.g. Stockfish) as soft targets, or
   self-play games.
4. **A policy head** that predicts the master's move. `replay_games` already
   returns `next_move` for every position.